# Go2 + Unitree Z1 Pick-and-Carry（Colab）

Genesis で Go2 の背中に Z1 を載せ、歩行 → 到達 → 持ち歩き → 箱運び / キャッチを学習します。観測 81・行動 17（脚 CPG、腕は IK+残差）。起動スクリプトと同じ `train_ppo`（必要なら `train_bc` / `train_gail`）を使います。

**先にランタイムを GPU にしてください。** 「ランタイム」→「ランタイムのタイプを変更」→ **T4** または **L4**。変更後は再起動して先頭から実行します。3D ウィンドウは使わず、結果は mp4（`videos/<タスク>/...`）で確認します。手順の詳細は [docs/COLAB.md](../docs/COLAB.md) です。


## Google Drive をマウント（任意・推奨）

リポジトリや `logs/` を Drive に置いている場合、または学習結果を退避したい場合に実行します。認証ダイアログが出たら許可してください。


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    raise SystemExit('このセルは Google Colab 向けです。')

MOUNT = '/content/drive'
if not Path(MOUNT).exists():
    drive.mount(MOUNT)
else:
    print('already mounted:', MOUNT)

for p in (
    Path('/content/drive/MyDrive/PAI_Task'),
    Path('/content/drive/MyDrive/content/PAI_Task'),
    Path('/content/drive/MyDrive/PAI_Task_logs'),
):
    print(('OK' if p.exists() else '--'), p)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['PYTHONUTF8'] = '1'

if shutil.which('nvidia-smi') is None:
    raise SystemExit(
        'このランタイムは CPU です。\n'
        '「ランタイム」→「ランタイムのタイプを変更」→ GPU にして、セッションを再起動してください。'
    )
subprocess.check_call(['nvidia-smi'])

# Drive 上のプロジェクト / 退避 logs を使うなら、先にマウントセルを実行する
if Path('/content').exists() and not Path('/content/drive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive マウントをスキップ:', exc)

candidates = [
    Path('/content/PAI_Task'),
    Path('/content/drive/MyDrive/PAI_Task'),
    Path('/content/drive/MyDrive/content/PAI_Task'),
    Path.cwd(),
]
root = next((path for path in candidates if (path / 'pick_and_carry').is_dir()), None)
if root is None:
    raise SystemExit(
        'pick_and_carry が見つかりません。\n'
        'zip を /content に展開するか、Drive の MyDrive/PAI_Task に置いてマウントセルを実行してください。'
    )
os.chdir(root)
print('cwd', Path.cwd())

# 以前 Drive に退避した logs があれば戻す（ローカルに無いときだけ）
drive_logs = Path('/content/drive/MyDrive/PAI_Task_logs/logs')
local_logs = Path('logs')
if drive_logs.is_dir() and not local_logs.exists():
    shutil.copytree(drive_logs, local_logs)
    print('restored logs from', drive_logs)


In [ ]:
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('PyTorch から GPU が見えません。ランタイムを GPU にして再起動してください。')
print('device', torch.cuda.get_device_name(0))


## 学習・推論の設定（起動スクリプトと同じ）

下の変数を変えてから、あとの学習セル / 推論セルを実行します。保存先はローカル起動と同じ `logs/{algo}_*` です。

起動メニュー対応: **1** 模倣（BC/GAIL）、**2** 学習、**3** 推論。行動は 17 次元（CPG+腕）。`EASY_WALK=True` の歩行は `*_walk_easy` に保存し、PPO が無いとき `gail_trot.pt` → `bc_trot.pt` を自動初期化します。


In [ ]:
from pathlib import Path
from pick_and_carry.algorithms import ALGORITHMS, ALGO_LABELS
from pick_and_carry.curriculum import CATCH_LEVELS, STAGES, stage_log_dir

# --- ここを起動スクリプトと同じ感覚で選ぶ ---
STAGE = 'walk'          # walk / reach / haul / carry / catch
CATCH_LEVEL = 'static'  # catch のときだけ: static / slow / random / fly / noisy
ALGO = 'ppo'            # ppo / grpo / sym_ppo

NUM_ENVS = 8            # T4 は 8 から。通ったら 32 まで。ローカル既定は 64
ITERATIONS = 50         # 追加で回す回数（--resume latest で続きから）
NUM_STEPS = 128         # 1 iteration あたりのステップ。起動スクリプトと同じ
SAVE_INTERVAL = 10
PLAY_SECONDS = 20.0     # 推論の録画秒数
EASY_WALK = True        # walk のとき True なら *_walk_easy（模倣暖機用）

# 箱運びだけ意味がある項目（他段階では train_ppo 側で上書きされることもあります）
CONTROL_SPLIT = 'split'     # split / walk / both
WRIST_CAMERA = False
TERRAIN_STEPS = False       # walk では地形あり。段差を足すなら True
TERRAIN_AMPLITUDE = 0.045   # TERRAIN_STEPS のとき使う高さ [m]
RETREAT = False             # True なら上の段階の重みで易しい課題を学び直す
# ------------------------------------------

assert STAGE in STAGES, STAGES
assert CATCH_LEVEL in CATCH_LEVELS, CATCH_LEVELS
assert ALGO in ALGORITHMS, ALGORITHMS

easy = bool(EASY_WALK and STAGE == 'walk')
LOG_DIR = stage_log_dir(
    STAGE,
    CATCH_LEVEL if STAGE == 'catch' else 'static',
    algo=ALGO,
    easy=easy,
)
LOG_DIR = Path(LOG_DIR)
VIDEO_NAME = {
    'carry': f'PickAndCarry_{ALGO}_z1.mp4',
    'catch': f'PickAndCarry_{ALGO}_catch_{CATCH_LEVEL}.mp4',
}.get(STAGE, f'PickAndCarry_{ALGO}_{STAGE}.mp4')
# 実体は logs/.../videos/<タスク>/<学習回>/ にも残る（--record はファイル名のヒント）
RECORD_PATH = LOG_DIR / VIDEO_NAME

print('手法', ALGO_LABELS[ALGO], '  段階', STAGE, end='')
if STAGE == 'catch':
    print(' /', CATCH_LEVEL)
else:
    print()
if easy:
    print('easy-walk', True)
print('保存先', LOG_DIR)
print('録画', RECORD_PATH)


## 観測と URDF の確認

観測 81、行動 17（CPG 10 + 腕残差 6 + グリッパ残差 1）、腕は Z1 のみです。脚は環境内 CPG、reach 以降の腕は IK ベース + 小さな残差です。


In [ ]:
from pick_and_carry.arms import get_arm
from pick_and_carry.config import NUM_ACTIONS, NUM_OBS, OBS_LAYOUT, layout_slices

arm = get_arm('z1')
print('obs', NUM_OBS, 'act', NUM_ACTIONS)
print('urdf', arm.urdf_path)
assert arm.urdf_path.is_file()
for name, sl in layout_slices(OBS_LAYOUT).items():
    print(f'{sl.start:02d}:{sl.stop:02d}  {name}')


## （任意）描画の煙テスト

1 体を数秒録画します。初回はカーネルコンパイルで数分かかります。飛ばして学習へ進んでも構いません。


In [ ]:
import torch
from IPython.display import Video, display
from pick_and_carry.config import EnvConfig
from pick_and_carry.env import PickAndCarryEnv

if not torch.cuda.is_available():
    raise SystemExit('GPU がありません。')

preview = 'logs/colab_preview.mp4'
env = PickAndCarryEnv(cfg=EnvConfig(
    num_envs=1,
    backend='gpu',
    show_viewer=False,
    record_path=preview,
    curriculum=STAGE,
    catch_level=CATCH_LEVEL if STAGE == 'catch' else 'static',
    arm='z1',
    seed=0,
))
print('genesis device', env.device)
if 'cpu' in str(env.device).lower():
    env.close()
    raise SystemExit('Genesis が CPU に落ちました。GPU ランタイムで再起動してください。')
obs = env.reset()
action = torch.zeros(env.num_envs, env.num_actions, device=obs.device)
for _ in range(100):
    obs, _, _, _ = env.step(action)
env.close()
print('video', preview)
display(Video(preview, embed=True, width=960))


## 学習

起動スクリプトの「2 学習」と同じです。`--resume latest` なので、同じ `LOG_DIR` に重みがあれば続きから、無ければ前段階の重みから始めます。`EASY_WALK` で PPO も `--init-from` も無いときは、同フォルダの `gail_trot.pt` → `bc_trot.pt` を自動で載せます。模倣だけ先に回すときはローカルメニュー 1、または `train_bc` / `train_gail` を使います。


In [ ]:
import shutil
import subprocess
import sys
import torch

if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
    raise SystemExit('GPU がありません。ランタイムを GPU にして再起動してください。')

cmd = [
    sys.executable, '-u', '-m', 'pick_and_carry.train_ppo',
    '--backend', 'gpu',
    '--arm', 'z1',
    '--algo', ALGO,
    '--curriculum', STAGE,
    '--num-envs', str(NUM_ENVS),
    '--iterations', str(ITERATIONS),
    '--num-steps', str(NUM_STEPS),
    '--save-interval', str(SAVE_INTERVAL),
    '--log-dir', str(LOG_DIR),
    '--resume', 'latest',
    '--control-split', CONTROL_SPLIT,
]
if STAGE == 'catch':
    cmd.extend(['--catch-level', CATCH_LEVEL])
if EASY_WALK and STAGE == 'walk':
    cmd.append('--easy-walk')
if RETREAT:
    cmd.append('--retreat')
if WRIST_CAMERA:
    cmd.append('--wrist-camera')
if TERRAIN_STEPS:
    cmd.extend(['--terrain-steps', '--terrain-amplitude', str(TERRAIN_AMPLITUDE)])

print(' '.join(cmd))
subprocess.check_call(cmd)
print('done', LOG_DIR)


## 推論（録画）

起動スクリプトの「3 推論」と同じく、保存した重みを 1 体で再生して mp4 を残します。ファイルは `videos/<タスク>/<学習回>/` にも分かれます。Colab ではクリック GUI や ROS は使わず、録画のみです。


In [ ]:
import subprocess
import sys
from IPython.display import Video, display
from pathlib import Path

RECORD_PATH.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-u', '-m', 'pick_and_carry.train_ppo',
    '--play', '--resume', 'latest',
    '--backend', 'gpu',
    '--arm', 'z1',
    '--algo', ALGO,
    '--curriculum', STAGE,
    '--num-envs', '1',
    '--seconds', str(PLAY_SECONDS),
    '--record', str(RECORD_PATH),
    '--log-dir', str(LOG_DIR),
    '--control-split', CONTROL_SPLIT,
]
if STAGE == 'catch':
    cmd.extend(['--catch-level', CATCH_LEVEL])
if EASY_WALK and STAGE == 'walk':
    cmd.append('--easy-walk')
if WRIST_CAMERA:
    cmd.append('--wrist-camera')
if TERRAIN_STEPS:
    cmd.extend(['--terrain-steps', '--terrain-amplitude', str(TERRAIN_AMPLITUDE)])

print(' '.join(cmd))
subprocess.check_call(cmd)
assert Path(RECORD_PATH).is_file(), RECORD_PATH
print('video', RECORD_PATH)
display(Video(str(RECORD_PATH), embed=True, width=960))


## （任意）学習曲線

metrics.jsonl があれば HTML を作ります。ノートからはファイルをダウンロードしてブラウザで開けます。


In [ ]:
import subprocess
import sys
from pathlib import Path

curves = Path('logs') / f'colab_curves_{ALGO}_{STAGE}.html'
cmd = [sys.executable, 'scripts/plot_curves.py', '--log-dir', str(LOG_DIR), '--output', str(curves)]
print(' '.join(cmd))
try:
    subprocess.check_call(cmd)
    print('curves', curves.resolve())
except subprocess.CalledProcessError:
    print('曲線を作れる記録がまだありません。先に学習セルを実行してください。')


## （任意）手法比較

同じ段階で PPO / GRPO / Symmetric PPO を学習済みなら、曲線を重ねます。


In [ ]:
import subprocess
import sys
from pathlib import Path
from pick_and_carry.curriculum import stage_log_dir

dirs = []
labels = []
for name in ('ppo', 'grpo', 'sym_ppo'):
    folder = stage_log_dir(STAGE, CATCH_LEVEL if STAGE == 'catch' else 'static', algo=name)
    if (folder / 'metrics.jsonl').is_file() or any(folder.glob('ppo_*.pt')):
        dirs.append(str(folder))
        labels.append(name)

if len(dirs) < 2:
    print('比較できる手法が 2 つ未満です。別の ALGO でも同じ STAGE を学習してください。')
else:
    out = Path('logs') / f'colab_compare_{STAGE}.html'
    cmd = [sys.executable, 'scripts/plot_curves.py', '--compare', *dirs, '--labels', *labels, '--output', str(out)]
    print(' '.join(cmd))
    subprocess.check_call(cmd)
    print('compare', out.resolve())


## セッションが切れる前に（Drive へ退避）

無料枠はディスクが消えます。続きを残すときは、このセルで `logs/` を Drive の `MyDrive/PAI_Task_logs` へコピーします。未マウントならここでマウントします。


In [ ]:
from pathlib import Path
import shutil

drive_root = Path('/content/drive')
if not drive_root.exists():
    try:
        from google.colab import drive
        drive.mount(str(drive_root))
    except Exception as exc:
        raise SystemExit(f'Drive をマウントできませんでした: {exc}')

drive_logs = Path('/content/drive/MyDrive/PAI_Task_logs')
drive_logs.mkdir(parents=True, exist_ok=True)
if Path('logs').is_dir():
    shutil.copytree('logs', drive_logs / 'logs', dirs_exist_ok=True)
    print('copied to', drive_logs / 'logs')
else:
    print('logs/ がありません')
